# Проект для "Викишоп" с BERT

__Описание проекта__

Интернет-магазин «Викишоп» запускает новый сервис. Теперь пользователи могут редактировать и дополнять описания товаров, как в вики-сообществах. То есть клиенты предлагают свои правки и комментируют изменения других. Магазину нужен инструмент, который будет искать токсичные комментарии и отправлять их на модерацию.

__Задачи проекта__

Обучите модель классифицировать комментарии на позитивные и негативные. В вашем распоряжении набор данных с разметкой о токсичности правок.

__Описание данных__
- `text` - содержит текст комментария
- `toxic` - целевой признак

__Требования__

Построить модель со значением метрики качества F1 не меньше 0.75.

__Выводы__

- Обучена модель классификации комментариев на позитивные и негативные.
- В данной работе используется `toxic_bert` (https://huggingface.co/unitary/toxic-bert). Так как он обучен на классификацию токсичных комментариев
- Для работы модели BERT необходимы тексты комментариев небольшого размера, вывод токенизатора должен составлять __небольше 512 токенов__ (510 + токен начала и конца текста) иначе __нельзя будет получить эмбеддинги__.
- Лучшая модель - __LightGBM__, с параметрами:
    - `learning_rate` = 0.05
    - `n_estimators` = 100
- При этом размер данных равен 1000
- На лучшей модели достигнуто значение метрики качества __F1 = 0.83__, что выше требуемого значения в 0.75

In [1]:
!pip install --upgrade -q scikit-learn
!pip install -q torch
!pip install -q transformers


[notice] A new release of pip is available: 23.2.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer, BertModel
from tqdm import notebook
from sklearn.metrics import f1_score, make_scorer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline

In [3]:
RANDOM_STATE = 42
MAX_LENGTH = 512
SIZE = 1000
PATH = ''

## Загрузка и подготовка данных

In [4]:
def load_df(filename, sep=',', decimal='.'):
    pth1 = "./datasets/" + filename
    pth2 = PATH + filename

    if os.path.exists(pth1):
        df = pd.read_csv(pth1, sep=sep, decimal=decimal)
    else:
        df = pd.read_csv(pth2, sep=sep, decimal=decimal)

    print('\n\tПервые строки датафрейма')
    display(df.head())

    print('\n\n\tИнформация о датафрейме\n')
    df.info()

    return df

In [5]:
data = load_df('toxic_comments.csv')


	Первые строки датафрейма


,Unnamed: 0,text,toxic
0,0,Explanation\nWhy the edits made under my usern...,0
1,1,D'aww! He matches this background colour I'm s...,0
2,2,"Hey man, I'm really not trying to edit war. It...",0
3,3,"""\nMore\nI can't make any real suggestions on ...",0
4,4,"You, sir, are my hero. Any chance you remember...",0




	Информация о датафрейме

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159292 entries, 0 to 159291
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   Unnamed: 0  159292 non-null  int64 
 1   text        159292 non-null  object
 2   toxic       159292 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.6+ MB


In [6]:
data = data[['text', 'toxic']]
data.head()

,text,toxic
0,Explanation\nWhy the edits made under my usern...,0
1,D'aww! He matches this background colour I'm s...,0
2,"Hey man, I'm really not trying to edit war. It...",0
3,"""\nMore\nI can't make any real suggestions on ...",0
4,"You, sir, are my hero. Any chance you remember...",0


In [7]:
data.isnull().sum()

text     0
toxic    0
dtype: int64

In [8]:
tokenizer = BertTokenizer.from_pretrained('unitary/toxic-bert')

data['token_text'] = data['text'].apply(lambda x: tokenizer.encode(x, add_special_tokens=True))
data['token_len'] = data['token_text'].apply(lambda x: len(x))

Token indices sequence length is longer than the specified maximum sequence length for this model (631 > 512). Running this sequence through the model will result in indexing errors


In [9]:
count = 0
for token in data['token_text'].values:
    if len(token) > 512:
        count += 1

print(f"Отношение текстов, длиной выше 512, ко всему количеству текстов = {count/len(data['token_text'])*100:.2f}%")

Отношение текстов, длиной выше 512, ко всему количеству текстов = 2.20%


Для этой модели BERT максимальная длина токенов 510 + 2 токена начала и конца строки.

Так как количество текстов больше 512 мало, можем их удалить

In [10]:
print(data.shape)
data = data[data['token_len'] <= 512]
data.shape

(159292, 4)


(155789, 4)

In [11]:
padded = np.array([token + [0]*(MAX_LENGTH - len(token)) for token in data['token_text'].values])
attention_mask = np.where(padded != 0, 1, 0)

In [12]:
# Проверка на то, что у каждого текста есть токен конца предложения
count = 0
for p, ind in zip(padded, data['token_len']):
    if p[ind - 1] == 102: # 102 - токен конца текста
        count += 1

count == len(padded)

True

In [13]:
model = BertModel.from_pretrained('unitary/toxic-bert')

In [14]:
padded = padded[:SIZE]
data = data[:SIZE]

In [15]:
padded.shape, data.shape

((1000, 512), (1000, 4))

In [16]:
batch_size = 100
embeddings = []
for i in notebook.tqdm(range(padded.shape[0] // batch_size)):
    batch = torch.LongTensor(padded[batch_size*i:batch_size*(i+1)])
    attention_mask_batch = torch.LongTensor(attention_mask[batch_size*i:batch_size*(i+1)])

    with torch.no_grad():
        batch_embeddings = model(batch, attention_mask=attention_mask_batch)

    embeddings.append(batch_embeddings[0][:,0,:].numpy())

  0%|          | 0/10 [00:00<?, ?it/s]

In [17]:
features = np.concatenate(embeddings)
features.shape

(1000, 768)

## Обучение моделей

In [18]:
models = {
    'log_reg': LogisticRegression(random_state=RANDOM_STATE),
    'random_forest': RandomForestClassifier(random_state=RANDOM_STATE),
    'lightgbm': LGBMClassifier(random_state=RANDOM_STATE, verbose=-1),
}

In [19]:
param_grid_rf = {
    'model': [models['random_forest']],
    'model__n_estimators': [100, 200],
    'model__max_depth': [10, 20]
}

In [20]:
param_grid_lgb = {
    'model': [models['lightgbm']],
    'model__n_estimators': [100, 200],
    'model__learning_rate': [0.05, 0.1]
}

In [21]:
pipeline = Pipeline(steps=[
    ('model', models['log_reg'])
])

In [22]:
all_params = [param_grid_rf, param_grid_lgb]

In [32]:
X_train, X_test, y_train, y_test = train_test_split(features,
                                                    data['toxic'].values,
                                                    test_size=.1,
                                                    random_state=RANDOM_STATE)

## Обучение

In [24]:
f1 = make_scorer(f1_score, average='binary')

In [33]:
search = GridSearchCV(pipeline, all_params, cv=5, scoring=f1)
search.fit(X_train, y_train)

D:\Programs\Anaconda3\envs\practicum_base\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
D:\Programs\Anaconda3\envs\practicum_base\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
D:\Programs\Anaconda3\envs\practicum_base\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
D:\Programs\Anaconda3\envs\practicum_base\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
D:\Programs\Anaconda3\envs\practicum_base\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifi

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('model',
                                        LogisticRegression(random_state=42))]),
             param_grid=[{'model': [RandomForestClassifier(random_state=42)],
                          'model__max_depth': [10, 20],
                          'model__n_estimators': [100, 200]},
                         {'model': [LGBMClassifier(random_state=42,
                                                   verbose=-1)],
                          'model__learning_rate': [0.05, 0.1],
                          'model__n_estimators': [100, 200]}],
             scoring=make_scorer(f1_score, response_method='predict', average=binary))

In [34]:
best_model = search.best_estimator_

print(f"Лучшая модель: {type(best_model.named_steps['model']).__name__}")
print(f"Лучшие параметры: {search.best_params_}")

Лучшая модель: LGBMClassifier
Лучшие параметры: {'model': LGBMClassifier(random_state=42, verbose=-1), 'model__learning_rate': 0.05, 'model__n_estimators': 100}


In [35]:
y_pred = best_model.predict(X_test)

D:\Programs\Anaconda3\envs\practicum_base\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [36]:
f1_score(y_test, y_pred, average='binary')

0.8333333333333334

## Итоговые выводы

- Обучена модель классификации комментариев на позитивные и негативные.
- В данной работе используется `toxic_bert` (https://huggingface.co/unitary/toxic-bert). Так как он обучен на классификацию токсичных комментариев
- Для работы модели BERT необходимы тексты комментариев небольшого размера, вывод токенизатора должен составлять __небольше 512 токенов__ (510 + токен начала и конца текста) иначе __нельзя будет получить эмбеддинги__.
- Лучшая модель - __LightGBM__, с параметрами:
    - `learning_rate` = 0.05
    - `n_estimators` = 100
- При этом размер данных равен 1000
- На лучшей модели достигнуто значение метрики качества __F1 = 0.83__, что выше требуемого значения в 0.75

С изменением размера данных (1000 -> 2000) увеличивается значение качества метрики F1 до 0.97

А также изменяется лучшая модель:
- `'model': RandomForestClassifier(random_state=42),`
- `'model__max_depth': 10,`
- `'model__n_estimators': 200`

Но время преобразования текстов в эмбеддинги увеличивается (примерно в 2 раза)